# Quick LPM training step on `dili_train_CL_0000182`

Smoke-test for the **modified (6-feature)** LPM. Trains on a single shard for a handful of steps to confirm that:

1. The local `lpm_style/perturb_lib` is the one being imported (not the upstream 3-feature fork).
2. `dili_train_CL_0000182` shards load via `OnDiskPlibData` and split cleanly into train/val.
3. `LPM.fit()` runs end-to-end: vocab init, encoding, forward, MSE loss, backward, optimizer step.
4. After a few steps every parameter is still **finite** (no NaN/Inf blowup).

Use this as the canary before launching the 50-GPU Slurm run.

## 1. Force the local `perturb_lib` fork on `sys.path`

Same trick as in `check_val_shards_nulls.ipynb`. Without this, the upstream 3-feature `perturb_lib` may resolve first and the model won't have `dataset_embedding_layer`, `log_dose_layer`, `time_layer`.

In [2]:
print(1)

1


In [3]:
import numpy as np

In [4]:
import inspect
import os
import sys

LPM_STYLE_ROOT = "/home/icb/olga.novitskaia/lpm_style"

for mod in [m for m in list(sys.modules) if m == "perturb_lib" or m.startswith("perturb_lib.")]:
    del sys.modules[mod]

sys.path = [LPM_STYLE_ROOT] + [p for p in sys.path if "perturblib" not in p and p != LPM_STYLE_ROOT]
os.chdir(LPM_STYLE_ROOT)

import perturb_lib as plib
from perturb_lib.models.collection.lpm import LPM
import perturb_lib.models.access as model_access

assert "lpm_style" in inspect.getfile(plib), f"perturb_lib resolved to {inspect.getfile(plib)}"
assert "lpm_style" in inspect.getfile(LPM), f"LPM resolved to {inspect.getfile(LPM)}"
model_access.model_catalogue["LPM"] = LPM

print("perturb_lib :", inspect.getfile(plib))
print("LPM class   :", inspect.getfile(LPM))

perturb_lib : /home/icb/olga.novitskaia/lpm_style/perturb_lib/__init__.py
LPM class   : /home/icb/olga.novitskaia/lpm_style/perturb_lib/models/collection/lpm.py


## 2. Load shards (single dataset or all)

Set `USE_ALL_DATASETS = False` to keep the fast canary on `dili_train_CL_0000182`.

Set `USE_ALL_DATASETS = True` to load **every** dataset listed under `on_disk_data_sources` in `perturb_gym/configs/collection/lpm_modified.yaml` — this reproduces the cluster training data exactly, but on a single process. Useful to:

- Stress-test the pipeline at full scale on one machine (slow but informative).
- Trigger NaN locally if a poisoned shard is the cause of the cluster blowup.

Train/val/test split mirrors `lpm_modified.yaml` (`val_and_test_perturbations_selected_from: "all"`).

In [22]:
from pathlib import Path
import polars as pl
import yaml

from perturb_lib.data.plibdata import OnDiskPlibData
from perturb_lib.data.access import split_plibdata_3fold

PLIBDATA_ROOT = Path(LPM_STYLE_ROOT) / ".plib_cache/plibdata"
CONFIG_PATH = Path(LPM_STYLE_ROOT) / "perturb_gym/configs/collection/lpm_modified.yaml"

USE_ALL_DATASETS = True   # flip to True to load every dataset from the YAML

if USE_ALL_DATASETS:
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    data_sources = cfg["data_configs"][0]["on_disk_data_sources"]
    print(f"Loading {len(data_sources)} datasets from {CONFIG_PATH.name}")
else:
    data_sources = ["dili_train_CL_0000182"]
    print(f"Loading single dataset: {data_sources[0]} (canary mode)")

all_data = OnDiskPlibData(
    data_sources=data_sources,
    path_to_data_sources=PLIBDATA_ROOT,
)

traindata, valdata, testdata = split_plibdata_3fold(all_data, context_ids=None)

print("train rows :", len(traindata))
print("val rows   :", len(valdata) if valdata is not None else None)
print("test rows  :", len(testdata) if testdata is not None else None)

Loading 160 datasets from lpm_modified.yaml
train rows : 1711397712
val rows   : 286998547
test rows  : 294692808


In [32]:
1711397712

PosixPath('/home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata')

## 3. Build LPM and run training

Calls `LPM.fit()` directly with a CPU/GPU-agnostic `Trainer` config.

- **Canary mode** (`USE_ALL_DATASETS = False`): runs 1 epoch over `dili_train_CL_0000182`. Quick smoke-test.
- **Full-data mode** (`USE_ALL_DATASETS = True`): caps batches per epoch via `limit_train_batches` so you can repro the cluster scenario in a tractable wall-clock. Adjust the cap if you want a longer run.

Notes:
- `accelerator: "auto"` + `devices: 1` → uses GPU if present, otherwise CPU.
- `gradient_clip_val=1.0` is the new default we'll also use in production; prevents the bf16 NaN blowup seen on the 50-GPU run.
- `precision="32-true"` here — easier to spot real issues; flip to `"bf16-mixed"` to reproduce the cluster precision.

In [40]:
import torch

torch.manual_seed(13)

# Cap per-epoch work when running over ALL datasets, to keep wall-clock reasonable
# on a single process. With ~3 B rows / 4000 batch ≈ 750 K steps per epoch, an
# unbounded run would take days. 200 batches × 4000 = 800 K samples per epoch —
# enough to surface NaN-blowup if a poisoned shard exists.
trainer_pars = {
    "accelerator": "auto",
    "devices": 1,
    "max_epochs": 1,
    "precision": "32-true",
    #"gradient_clip_val": 1.0,
    #"gradient_clip_algorithm": "norm",
    "enable_checkpointing": False,
    "logger": False,
    "enable_progress_bar": True,
    "deterministic": True,
    "num_sanity_val_steps": 0,
}
#if USE_ALL_DATASETS:
#    trainer_pars["limit_train_batches"] = 200
#    trainer_pars["limit_val_batches"] = 50

model = LPM(
    embedding_dim=128,
    optimizer_name="AdamW",
    learning_rate=2e-3,
    learning_rate_decay=0.97,
    num_layers=2,
    hidden_dim=256,
    dropout=0.1,
    batch_size=16000,
    num_workers=0,
    pin_memory=False,
    early_stopping_patience=0,
    lightning_trainer_pars=trainer_pars,
)

model.fit(traindata=traindata, valdata=valdata)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
14:35:23 | INFO | Fitting LPM..

  | Name                    | Type         | Params | Mode 
-----------------------------------------------------------------
0 | loss                    | MSELoss      | 0      | train
1 | predictor               | Sequential   | 262 K  | train
2 | dataset_embedding_layer | Embedding    | 1.3 K  | train
3 | context_embedding_layer | Embedding    | 16.6 K | train
4 | perturb_embedding_layer | EmbeddingBag | 4.8 M  | train
5 | readout_embedding_layer | Embedding    | 5.6 M  | train
6 | log_dose_layer          | Linear       | 256    | train
7 | time_layer              | Linear       | 256    | train
-----------------------------------------------------------------
10.6 M    Trainable params
0         Non-trainable params
10.6 M    Total params
42.567    Total estimated model params size (MB)
15        Modules in train mode
0         Modules in e

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [41]:
df_pert = model.vocab.perturb_vocab.to_pandas()

In [42]:
embeddings = model.perturb_embedding_layer.weight.detach().numpy().astype(np.float64)

In [43]:
embeddings

array([[-0.84051138,  1.11704695,  1.31227303, ..., -0.45909381,
        -0.35053304,  0.85126662],
       [-0.25088143, -1.20581281, -0.49967194, ...,  1.64362502,
         0.36248374,  1.81782496],
       [-0.15835837,  0.20713712,  0.75161439, ..., -1.25357831,
        -0.67245686,  0.08602814],
       ...,
       [-0.88610387, -0.07415893, -0.66141641, ...,  0.64491373,
         0.96497089,  0.35614237],
       [-0.61309212, -0.9778583 , -0.23645972, ..., -0.37953898,
        -0.82365483,  1.18463922],
       [-1.46191335,  0.30048847,  0.73423475, ..., -2.05014539,
        -1.77085125, -0.0253739 ]])

In [44]:
np.isnan(embeddings).any()

False

## 4. Confirm no NaN / Inf in any parameter after training

If this prints `All parameters finite.` the smoke-test passed. If anything shows up, training already started to diverge in 20 steps — we'd need to lower the LR further or check the dataset shards for non-finite values.

In [38]:
bad = []
for name, p in model.named_parameters():
    n_nan = torch.isnan(p).sum().item()
    n_inf = torch.isinf(p).sum().item()
    if n_nan or n_inf:
        bad.append((name, tuple(p.shape), n_nan, n_inf, p.numel()))
        print(f"  {name:40s} shape={tuple(p.shape)} NaN={n_nan} Inf={n_inf} total={p.numel()}")

if not bad:
    print("All parameters finite.")
else:
    print(f"\n{len(bad)} parameter tensor(s) contain NaN/Inf — training already diverged.")

All parameters finite.


## 5. Sanity predict on a tiny validation slice

Final check: predictions are finite numbers, not NaN. This is exactly the failure mode that caused `ValueError: Input contains NaN` in `evaluate_model`.

In [39]:
import numpy as np

if valdata is None or len(valdata) == 0:
    print("No val data — skipping predict.")
else:
    val_sample = valdata.subset(slice(0, 1024))
    cols = ["dataset", "context", "perturbation", "readout", "log_dose", "time"]
    y_pred = model.predict(val_sample.subset_columnwise(cols))

    y_true_df = val_sample._data.collect() if hasattr(val_sample, "_data") else None
    y_true = y_true_df["value"].to_numpy() if y_true_df is not None else None

    print("y_pred shape   :", y_pred.shape)
    print("y_pred finite  :", int(np.isfinite(y_pred).sum()), "/", y_pred.size)
    print("y_pred NaN     :", int(np.isnan(y_pred).sum()))
    if y_true is not None:
        print("y_true finite  :", int(np.isfinite(y_true).sum()), "/", y_true.size)
        print("y_true NaN     :", int(np.isnan(y_true).sum()))

AttributeError: 'OnDiskPlibData' object has no attribute 'subset'